In [1]:
import os
import mlflow
from ultralytics import YOLO

In [2]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "TQanh23"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "4893e77fec7d32ca9090e4d472ec706354aedc07"
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/TQanh23/ppe_v2.mlflow"

In [3]:
RUN_ID = "2526e8798a5f41f282910a213d8ff073" 

print(f"Downloading weights from DagsHub (Run ID: {RUN_ID})...")

In [6]:
download_dir = r"D:\ppe_cv\ppe_v2\downloads"
os.makedirs(download_dir, exist_ok=True) 

# 4. Download the artifact to the specified path
downloaded_model_path = mlflow.artifacts.download_artifacts(
    run_id=RUN_ID,
    artifact_path="weights/last.pt",
    dst_path=download_dir  # MLflow will save the file inside this folder
)

print(f"✅ Successfully downloaded to: {downloaded_model_path}")

✅ Successfully downloaded to: D:\ppe_cv\ppe_v2\downloads\last.pt


In [7]:
model = YOLO(downloaded_model_path)

In [8]:
print(model.model.is_fused())

False


In [9]:
import torch.nn as nn

bn_count = 0

for name, module in model.model.named_modules():
    if isinstance(module, nn.BatchNorm2d):
        bn_count += 1
        print(name, module.num_features)

print("\nTotal BatchNorm:", bn_count)

model.0.bn 48
model.1.bn 96
model.2.cv1.bn 96
model.2.cv2.bn 96
model.2.m.0.cv1.bn 48
model.2.m.0.cv2.bn 48
model.2.m.1.cv1.bn 48
model.2.m.1.cv2.bn 48
model.3.bn 192
model.4.cv1.bn 192
model.4.cv2.bn 192
model.4.m.0.cv1.bn 96
model.4.m.0.cv2.bn 96
model.4.m.1.cv1.bn 96
model.4.m.1.cv2.bn 96
model.4.m.2.cv1.bn 96
model.4.m.2.cv2.bn 96
model.4.m.3.cv1.bn 96
model.4.m.3.cv2.bn 96
model.5.bn 384
model.6.cv1.bn 384
model.6.cv2.bn 384
model.6.m.0.cv1.bn 192
model.6.m.0.cv2.bn 192
model.6.m.1.cv1.bn 192
model.6.m.1.cv2.bn 192
model.6.m.2.cv1.bn 192
model.6.m.2.cv2.bn 192
model.6.m.3.cv1.bn 192
model.6.m.3.cv2.bn 192
model.7.bn 576
model.8.cv1.bn 576
model.8.cv2.bn 576
model.8.m.0.cv1.bn 288
model.8.m.0.cv2.bn 288
model.8.m.1.cv1.bn 288
model.8.m.1.cv2.bn 288
model.9.cv1.bn 288
model.9.cv2.bn 576
model.12.cv1.bn 384
model.12.cv2.bn 384
model.12.m.0.cv1.bn 192
model.12.m.0.cv2.bn 192
model.12.m.1.cv1.bn 192
model.12.m.1.cv2.bn 192
model.15.cv1.bn 192
model.15.cv2.bn 192
model.15.m.0.cv1.bn 96


In [10]:
import torch

all_gamma = []

for name, module in model.model.named_modules():

    if isinstance(module, torch.nn.BatchNorm2d):

        all_gamma.append(module.weight.data.abs().clone())

all_gamma = torch.cat(all_gamma)

print("="*60)
print("GLOBAL BN STATISTICS")
print("="*60)

print("Total Channels :", len(all_gamma))
print("Min Gamma      :", all_gamma.min().item())
print("Max Gamma      :", all_gamma.max().item())
print("Mean Gamma     :", all_gamma.mean().item())
print("Std Gamma      :", all_gamma.std().item())

GLOBAL BN STATISTICS
Total Channels : 16560
Min Gamma      : 0.1512451171875
Max Gamma      : 10.0078125
Mean Gamma     : 1.086721420288086
Std Gamma      : 0.4372999966144562


In [13]:
prune_ratio = 0.3

sorted_gamma, _ = torch.sort(all_gamma)

threshold_index = int(len(sorted_gamma) * prune_ratio)

threshold = sorted_gamma[threshold_index]

print("="*60)
print(f"Prune Ratio : {prune_ratio*100:.0f}%")
print(f"Threshold   : {threshold:.6f}")

Prune Ratio : 30%
Threshold   : 0.891113


In [14]:
import torch
import torch.nn as nn

mask_dict = {}

print("=" * 80)
print("GENERATE PRUNING MASK")
print("=" * 80)

total_channels = 0
remain_channels = 0

for name, module in model.model.named_modules():

    # Chỉ xử lý BatchNorm
    if not isinstance(module, nn.BatchNorm2d):
        continue

    # Bỏ qua Detect Head (model.22)
    if name.startswith("model.22"):
        continue

    weight = module.weight.data.abs()

    mask = weight >= threshold

    # Không cho phép layer bị prune sạch
    if mask.sum() == 0:
        max_index = torch.argmax(weight)
        mask[max_index] = True

    mask_dict[name] = mask

    remain = int(mask.sum())
    total = len(mask)

    total_channels += total
    remain_channels += remain

    print(f"{name:<35} {remain:4d}/{total:4d}")

print("=" * 80)
print(f"Original Channels : {total_channels}")
print(f"Remaining Channels: {remain_channels}")
print(f"Pruned Channels   : {total_channels - remain_channels}")
print(f"Actual Pruning    : {(total_channels - remain_channels) / total_channels * 100:.2f}%")

GENERATE PRUNING MASK
model.0.bn                            48/  48
model.1.bn                            96/  96
model.2.cv1.bn                        71/  96
model.2.cv2.bn                        56/  96
model.2.m.0.cv1.bn                    46/  48
model.2.m.0.cv2.bn                    37/  48
model.2.m.1.cv1.bn                    42/  48
model.2.m.1.cv2.bn                    48/  48
model.3.bn                            13/ 192
model.4.cv1.bn                       100/ 192
model.4.cv2.bn                       123/ 192
model.4.m.0.cv1.bn                    51/  96
model.4.m.0.cv2.bn                     5/  96
model.4.m.1.cv1.bn                    44/  96
model.4.m.1.cv2.bn                    13/  96
model.4.m.2.cv1.bn                     7/  96
model.4.m.2.cv2.bn                    42/  96
model.4.m.3.cv1.bn                     7/  96
model.4.m.3.cv2.bn                    96/  96
model.5.bn                           185/ 384
model.6.cv1.bn                       202/ 384
model.6.cv2.

In [15]:
channel_cfg = {}

print("=" * 80)
print("CHANNEL CONFIGURATION")
print("=" * 80)

for name, mask in mask_dict.items():

    channel_cfg[name] = int(mask.sum())

    print(f"{name:<35} -> {channel_cfg[name]}")

CHANNEL CONFIGURATION
model.0.bn                          -> 48
model.1.bn                          -> 96
model.2.cv1.bn                      -> 71
model.2.cv2.bn                      -> 56
model.2.m.0.cv1.bn                  -> 46
model.2.m.0.cv2.bn                  -> 37
model.2.m.1.cv1.bn                  -> 42
model.2.m.1.cv2.bn                  -> 48
model.3.bn                          -> 13
model.4.cv1.bn                      -> 100
model.4.cv2.bn                      -> 123
model.4.m.0.cv1.bn                  -> 51
model.4.m.0.cv2.bn                  -> 5
model.4.m.1.cv1.bn                  -> 44
model.4.m.1.cv2.bn                  -> 13
model.4.m.2.cv1.bn                  -> 7
model.4.m.2.cv2.bn                  -> 42
model.4.m.3.cv1.bn                  -> 7
model.4.m.3.cv2.bn                  -> 96
model.5.bn                          -> 185
model.6.cv1.bn                      -> 202
model.6.cv2.bn                      -> 343
model.6.m.0.cv1.bn                  -> 168
model.6.m

In [16]:
import torch

torch.save(channel_cfg, "channel_cfg_30.pt")

print("Saved channel_cfg_30.pt")

Saved channel_cfg_30.pt


In [17]:
torch.save(mask_dict, "mask_dict_30.pt")

print("Saved mask_dict_30.pt")

Saved mask_dict_30.pt


In [18]:
print("="*80)
print("LAYER DEPENDENCY")
print("="*80)

for i, m in enumerate(model.model.model):

    print(f"[{i}] {m.__class__.__name__}")

    if hasattr(m, "f"):
        print("  From :", m.f)

    if hasattr(m, "i"):
        print("  Index:", m.i)

    print()

LAYER DEPENDENCY
[0] Conv
  From : -1
  Index: 0

[1] Conv
  From : -1
  Index: 1

[2] C2f
  From : -1
  Index: 2

[3] Conv
  From : -1
  Index: 3

[4] C2f
  From : -1
  Index: 4

[5] Conv
  From : -1
  Index: 5

[6] C2f
  From : -1
  Index: 6

[7] Conv
  From : -1
  Index: 7

[8] C2f
  From : -1
  Index: 8

[9] SPPF
  From : -1
  Index: 9

[10] Upsample
  From : -1
  Index: 10

[11] Concat
  From : [-1, 6]
  Index: 11

[12] C2f
  From : -1
  Index: 12

[13] Upsample
  From : -1
  Index: 13

[14] Concat
  From : [-1, 4]
  Index: 14

[15] C2f
  From : -1
  Index: 15

[16] Conv
  From : -1
  Index: 16

[17] Concat
  From : [-1, 12]
  Index: 17

[18] C2f
  From : -1
  Index: 18

[19] Conv
  From : -1
  Index: 19

[20] Concat
  From : [-1, 9]
  Index: 20

[21] C2f
  From : -1
  Index: 21

[22] Detect
  From : [15, 18, 21]
  Index: 22



In [19]:
print("="*80)
print("MODEL SAVE CONNECTION")
print("="*80)

for i, m in enumerate(model.model.model):

    print(
        f"Layer {i:2d} | "
        f"{m.__class__.__name__:<10} | "
        f"save={getattr(m,'save',None)} | "
        f"from={getattr(m,'f',None)}"
    )

MODEL SAVE CONNECTION
Layer  0 | Conv       | save=None | from=-1
Layer  1 | Conv       | save=None | from=-1
Layer  2 | C2f        | save=None | from=-1
Layer  3 | Conv       | save=None | from=-1
Layer  4 | C2f        | save=None | from=-1
Layer  5 | Conv       | save=None | from=-1
Layer  6 | C2f        | save=None | from=-1
Layer  7 | Conv       | save=None | from=-1
Layer  8 | C2f        | save=None | from=-1
Layer  9 | SPPF       | save=None | from=-1
Layer 10 | Upsample   | save=None | from=-1
Layer 11 | Concat     | save=None | from=[-1, 6]
Layer 12 | C2f        | save=None | from=-1
Layer 13 | Upsample   | save=None | from=-1
Layer 14 | Concat     | save=None | from=[-1, 4]
Layer 15 | C2f        | save=None | from=-1
Layer 16 | Conv       | save=None | from=-1
Layer 17 | Concat     | save=None | from=[-1, 12]
Layer 18 | C2f        | save=None | from=-1
Layer 19 | Conv       | save=None | from=-1
Layer 20 | Concat     | save=None | from=[-1, 9]
Layer 21 | C2f        | save=None